# Cross-Encoder Demo for Medical Term Re-ranking

This notebook demonstrates three different approaches to using the cross-encoder for medical term re-ranking:
1. Interactive input mode
2. Single test description
3. Batch processing from CSV

In [1]:
import pandas as pd
from encoder import MedicalTermRanker, candidate_pool

d:\Repos\ocl-sandbox\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Interactive Input Mode
This cell implements the interactive mode where users can type descriptions and see results in real-time.

In [2]:
ranker = MedicalTermRanker(candidate_pool=candidate_pool, method='cross')

def interactive_mode():
    print("Enter medical descriptions (type 'quit' to exit):")
    while True:
        description = input("\nEnter description: ").strip()
        if description.lower() == 'quit':
            break
        if not description:
            print("Please enter a valid description.")
            continue
        
        ranked_results = ranker.rank_terms(description)
        ranker.print_ranked_results(ranked_results)

interactive_mode()

Enter medical descriptions (type 'quit' to exit):
Please enter a valid description.


## 2. Single Test Description
This cell demonstrates using a single hard-coded test description.

In [3]:
description="blood glucose measurement"

In [4]:
cross_ranker = MedicalTermRanker(candidate_pool=candidate_pool, method='cross')
ranked_results = cross_ranker.rank_terms(description)
cross_ranker.print_ranked_results(ranked_results)

Ranking terms: 100%|██████████| 20/20 [00:00<00:00, 139.78it/s]


Ranked Results:
LOINC Code                                                             Description    Score
    2345-7                                Glucose [Mass/volume] in Serum or Plasma   0.3159
    4544-3                Hematocrit [Volume Fraction] of Blood by Automated count  -8.0850
     718-7                                       Hemoglobin [Mass/volume] in Blood  -8.1794
    1975-2                        Total Bilirubin [Mass/volume] in Serum or Plasma  -9.3607
    2571-8                           Triglyceride [Mass/volume] in Serum or Plasma  -9.5033
    2093-3                            Cholesterol [Mass/volume] in Serum or Plasma  -9.6174
     751-8                      Neutrophils [#/volume] in Blood by Automated count  -9.7745
    2085-9                        HDL Cholesterol [Mass/volume] in Serum or Plasma  -9.9587
    2089-1                        LDL Cholesterol [Mass/volume] in Serum or Plasma  -9.9724
     777-3                        Platelets [#/volume] in Blood

In [5]:
bi_ranker = MedicalTermRanker(candidate_pool=candidate_pool, method='bi')
ranked_results = bi_ranker.rank_terms(description)
bi_ranker.print_ranked_results(ranked_results)

Ranking terms: 100%|██████████| 20/20 [00:00<00:00, 75.12it/s]


Ranked Results:
LOINC Code                                                             Description  Score
    2345-7                                Glucose [Mass/volume] in Serum or Plasma 0.7198
     718-7                                       Hemoglobin [Mass/volume] in Blood 0.4248
    2571-8                           Triglyceride [Mass/volume] in Serum or Plasma 0.4228
    4544-3                Hematocrit [Volume Fraction] of Blood by Automated count 0.3776
    1920-8                      AST [Enzymatic activity/volume] in Serum or Plasma 0.3458
    2160-0                             Creatinine [Mass/volume] in Serum or Plasma 0.3367
    3094-0                                    BUN [Mass/volume] in Serum or Plasma 0.3328
    2093-3                            Cholesterol [Mass/volume] in Serum or Plasma 0.3324
     731-0                      Lymphocytes [#/volume] in Blood by Automated count 0.3274
    1975-2                        Total Bilirubin [Mass/volume] in Serum or Plasma 

## 3. Batch Processing from CSV
This cell demonstrates processing multiple descriptions from a CSV file and analyzing the results.

In [7]:
def process_csv(ranker, csv_path='test_descriptions.csv'):
    # Read the CSV file
    df = pd.read_csv(csv_path)
    
    # Process each description
    results = []
    for idx, row in df.iterrows():
       
        description = row['description']
        expected_loinc = row['expected_loinc']
        
        print(f"\nProcessing: {description}")
        ranked_results = ranker.rank_terms(description)
        
        # Get the top result
        top_result = ranked_results[0]
        
        # Store results
        results.append({
            'description': description,
            'expected_loinc': expected_loinc,
            'top_loinc': top_result[0],
            'top_score': top_result[2],
            'match': expected_loinc == top_result[0]
        })
        
        # Print individual results
        print(f"Expected LOINC: {expected_loinc}")
        ranker.print_ranked_results([top_result])
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print(f"Total tests: {len(results_df)}")
    print(f"Correct matches: {results_df['match'].sum()}")
    print(f"Accuracy: {(results_df['match'].sum() / len(results_df)) * 100:.2f}%")
    
    return results_df

In [8]:
df_cross_result = process_csv(cross_ranker)


Processing: blood sugar test


Ranking terms: 100%|██████████| 20/20 [00:00<00:00, 151.48it/s]


Expected LOINC: 2345-7

Ranked Results:
LOINC Code                              Description   Score
    2345-7 Glucose [Mass/volume] in Serum or Plasma -2.4427

Processing: complete blood count with hemoglobin


Ranking terms: 100%|██████████| 20/20 [00:00<00:00, 152.18it/s]


Expected LOINC: 718-7

Ranked Results:
LOINC Code                       Description   Score
     718-7 Hemoglobin [Mass/volume] in Blood -2.4621

Processing: liver function test


...

Expected LOINC: 1920-8

Ranked Results:
LOINC Code                           Description    Score
     785-6 MCH [Entitic mass] by Automated count -11.1695

Processing: cholesterol panel


Ranking terms: 100%|██████████| 20/20 [00:00<00:00, 153.24it/s]


Expected LOINC: 2093-3

Ranked Results:
LOINC Code                                  Description   Score
    2093-3 Cholesterol [Mass/volume] in Serum or Plasma -2.5804

Processing: kidney function test


Ranking terms: 100%|██████████| 20/20 [00:00<00:00, 161.66it/s]

Expected LOINC: 2160-0

Ranked Results:
LOINC Code                                        Description    Score
     751-8 Neutrophils [#/volume] in Blood by Automated count -11.0727

Summary Statistics:
Total tests: 5
Correct matches: 3
Accuracy: 60.00%


In [9]:
df_bi_result = process_csv(bi_ranker)


Processing: blood sugar test


Ranking terms: 100%|██████████| 20/20 [00:00<00:00, 78.52it/s]


Expected LOINC: 2345-7

Ranked Results:
LOINC Code                              Description  Score
    2345-7 Glucose [Mass/volume] in Serum or Plasma 0.5139

Processing: complete blood count with hemoglobin


Ranking terms: 100%|██████████| 20/20 [00:00<00:00, 74.89it/s]


Expected LOINC: 718-7

Ranked Results:
LOINC Code                       Description  Score
     718-7 Hemoglobin [Mass/volume] in Blood 0.6898

Processing: liver function test


...

Expected LOINC: 1920-8

Ranked Results:
LOINC Code                                                             Description  Score
    1742-6 Alanine aminotransferase [Enzymatic activity/volume] in Serum or Plasma 0.2902

Processing: cholesterol panel


Ranking terms: 100%|██████████| 20/20 [00:00<00:00, 85.08it/s]


Expected LOINC: 2093-3

Ranked Results:
LOINC Code                                  Description  Score
    2093-3 Cholesterol [Mass/volume] in Serum or Plasma 0.5843

Processing: kidney function test


Ranking terms: 100%|██████████| 20/20 [00:00<00:00, 85.57it/s]

Expected LOINC: 2160-0

Ranked Results:
LOINC Code                                 Description  Score
    2160-0 Creatinine [Mass/volume] in Serum or Plasma 0.3734

Summary Statistics:
Total tests: 5
Correct matches: 4
Accuracy: 80.00%
